# 05. PEFT Representation Finetuning: LoReFT

- **Category:** 15_compression_peft_and_speculative_inference / 03_parameter_efficient_finetuning_peft
- **Lineage & Citations:** Wu et al. (Stanford 2024, *ReFT: Representation Finetuning for Language Models*).
- **Purpose:** Intervene directly on hidden representations via low-rank orthogonal subspaces ($\Phi(h) = h + R^\top(W h + b - R h)$) at selective token positions, achieving state-of-the-art adaptation with up to $50\times$ fewer parameters than LoRA.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [PEFT Adapters Ia3 Prefix Prompt and Llama Adapter](04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.ipynb) $\to$ `🔗 Downstream Syntheses:` [Category 15 Index](../../../README.md#15_compression_peft_and_speculative_inference)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use LoReFT when fine-tuning requires fundamental pre-training knowledge rewriting or dense parametric memorization (where full fine-tuning or weight-merged LoRA is required). LoReFT steers representations at targeted token positions.

## Mathematical Foundations & Governing Equations

$$
\Delta s = (W h + b) - R h, \quad \Delta h = R^\top \Delta s, \quad h_{\text{out}} = h + \text{mask} \odot \Delta h, \quad R R^\top = I_r
$$

### Asymptotic Time & Memory Complexity
- **Trainable Parameters:** $2 \cdot r \cdot d + r$ per intervention site ($r \in \{1, 2, 4\}$, $< 0.002\%$ total parameters).
- **FLOPs Overhead:** $\mathcal{O}(B \cdot S_{\text{masked}} \cdot r \cdot d)$, negligible relative to dense transformer layer projections.

In [ ]:
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/05_peft_representation_finetuning_loreft.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def loreft_subspace_delta(h: torch.Tensor, r_mat: torch.Tensor, w_mat: torch.Tensor, b_vec: torch.Tensor) -> torch.Tensor:
    proj_r = torch.matmul(h, r_mat.t())
    proj_w = torch.matmul(h, w_mat.t()) + b_vec
    delta_s = proj_w - proj_r
    return torch.matmul(delta_s, r_mat)

def loreft_intervene(h: torch.Tensor, r_mat: torch.Tensor, w_mat: torch.Tensor, b_vec: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    delta_h = loreft_subspace_delta(h, r_mat, w_mat, b_vec)
    return h + mask.unsqueeze(-1) * delta_h

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

delta_h = loreft_subspace_delta(weights["h"], weights["r_matrix"], weights["w_matrix"], weights["b_vec"])
assert float(torch.max(torch.abs(delta_h - weights["delta_h"]))) < 1e-5

loreft_out = loreft_intervene(weights["h"], weights["r_matrix"], weights["w_matrix"], weights["b_vec"], weights["position_mask"])
assert float(torch.max(torch.abs(loreft_out - weights["loreft_out"]))) < 1e-5
print("All LoReFT representation intervention parity checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: loreft_subspace_delta(weights["h"], weights["r_matrix"], weights["w_matrix"], weights["b_vec"]), iters=50, name="LoReFT Subspace Delta"),
    benchmark_fn(lambda: loreft_intervene(weights["h"], weights["r_matrix"], weights["w_matrix"], weights["b_vec"], weights["position_mask"]), iters=50, name="LoReFT Intervene (Full)"),
]
print(format_benchmark_table(benchmarks))

In [ ]:
# Pathological Misuse: Unconstrained full-space drift vs LoReFT Orthogonal Subspace Invariance
h, r_mat = weights["h"], weights["r_matrix"]
d_dim = h.shape[-1]
eye = torch.eye(d_dim)
proj_subspace = r_mat.t() @ r_mat
proj_orthogonal = eye - proj_subspace

# LoReFT shift delta_h lies strictly in the subspace spanned by rows of R:
delta_loreft = weights["delta_h"]
leakage_loreft = float(torch.norm(delta_loreft @ proj_orthogonal))

# Arbitrary unconstrained activation shift causes severe collateral representation leakage:
delta_unconstrained = torch.randn_like(delta_loreft) * 0.1
leakage_unconstrained = float(torch.norm(delta_unconstrained @ proj_orthogonal))

print(f"LoReFT Orthogonal Leakage ||Delta @ (I - R^T R)||: {leakage_loreft:.2e}")
print(f"Unconstrained Leakage ||Delta_uncons @ (I - P)||:   {leakage_unconstrained:.4f}")
assert leakage_loreft < 1e-5 and leakage_unconstrained > 0.1, "Pathology demonstrated!"


## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Causal Representation Steering:** Intervening directly on activations bypasses weight update overhead, controlling downstream semantics through direct subspace shifts.
2. **Strict Subspace Isolation:** Because $\Delta h = R^\top \Delta s$, perturbations are orthogonal to $(I - R^\top R)$, provably protecting all unrelated representational features from collateral damage.
3. **Extreme Parameter Efficiency:** Achieving high downstream accuracy with as few as $\approx 0.002\%$ trainable parameters unlocks high-density multi-task deployment on resource-constrained devices.

---
### Cross-Domain Generalisation Matrix
| Domain | Subspace Intervention Analogue | Practical Impact |
| :--- | :--- | :--- |
| **LLM Steering & Safety** | Concept vector clamping | Eliminates toxic or out-of-distribution outputs at specific token positions |
| **Multi-Task Prompting** | Task-specific subspace projections | Swaps active task state via tiny vector offsets without weight reload |
| **Robotics & Policy Control** | Latent action space steering | Restricts policy interventions to safe, kinematically valid action manifolds |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_peft_representation_finetuning_loreft.md](../../../markdown/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/05_peft_representation_finetuning_loreft.md)
- **PyTorch 2.x Module:** [05_peft_representation_finetuning_loreft.py](../../../pytorch2/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/05_peft_representation_finetuning_loreft.py)
- **Functional JAX Module:** [05_peft_representation_finetuning_loreft.py](../../../jax/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/05_peft_representation_finetuning_loreft.py)
- **Native Rust Candle Track:** [05_peft_representation_finetuning_loreft.rs](../../../candle/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/05_peft_representation_finetuning_loreft.rs)
- **Hardware Kernel (CPU / WGSL):** [05_peft_representation_finetuning_loreft.rs](../../../kernels/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/05_peft_representation_finetuning_loreft.rs)
- **Golden SafeTensors:** [05_peft_representation_finetuning_loreft.safetensors](../../../weights/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/05_peft_representation_finetuning_loreft.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 04. PEFT Adapters Ia3 Prefix Prompt and Llama Adapter](04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.ipynb) | [Master Index](../../../README.md#15_compression_peft_and_speculative_inference) | Next: [Category 15 Index](../../../README.md#15_compression_peft_and_speculative_inference) &rarr;